In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

/backup/workspace/github/agentic-ai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00,   "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}


@tool
def get_product(name: str) -> str:
    """ Look up a product by name and return its price, rating, stock, and description. """
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {','.join(PRODUCTS)}"
    return str(p)


@tool
def get_review(name: str) -> str:
    """ Look up a product review by a product name and return the product name, number of reviews and rating """
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not found. Available: {','.join(REVIEWS)}"
    return str(r)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
# In memory 
llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent2= create_agent(
    llm_groq,
    tools=[get_product,get_review],
    system_prompt="You are a helpful product assistant for an online tech store."
)

agent3=create_agent(
    llm_groq,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
    checkpointer=InMemorySaver()
)

In [4]:
def ask3(question: str):
    config={"configurable":{"thread_id":"user-alice-session-1"}}
    result=agent3.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config
    )
    print(result["messages"][-1].content)

In [5]:
ask3("where is the capital city of nepal")
ask3("what is the prices of smart watch?")

The capital city of Nepal is Kathmandu.
The price of the smart watch is $199.99. It tracks heart rate and sleep, has a 5-day battery, and is water-resistant.


In [6]:
ask3("what about india?")
ask3("what about reviews of this product?")

The smart watch has a price of $199.99, tracks heart rate and sleep, has a 5-day battery, and is water-resistant. It has 340 reviews with an average rating of 3.9.
The smart watch has 340 reviews with an average rating of 3.9.
